# 🧪 BlogMS — Posts Tests (`test_posts.py`)

| # | Test | الوصف |
|---|------|-------|
| 1 | `test_list_posts_public` | جلب البوستات بدون توكن |
| 2 | `test_create_post_as_author` | Author ينشر بوست |
| 3 | `test_create_post_as_reader_forbidden` | Reader ممنوع من النشر |
| 4 | `test_create_post_unauthenticated` | نشر بدون توكن |
| 5 | `test_get_single_post` | جلب بوست بالـ ID |
| 6 | `test_get_nonexistent_post` | جلب بوست مش موجود |
| 7 | `test_update_own_post` | Author يعدّل بوسته |
| 8 | `test_delete_post_as_admin` | Admin يحذف أي بوست |
| 9 | `test_delete_post_as_non_admin_forbidden` | Author ممنوع من الحذف |
| 10 | `test_pagination` | Pagination صح مع 5 بوستات |

## 📋 `test_list_posts_public`

In [ ]:
def test_list_posts_public(client):
    """✅ 200 · items موجود · total=0"""
    resp = client.get("/posts/")
    assert resp.status_code == 200
    assert "items" in resp.json()
    assert resp.json()["total"] == 0

## ✍️ `test_create_post_as_author`

In [ ]:
def test_create_post_as_author(client, author_token):
    """✅ 201 · title صح · author.username=authoruser"""
    resp = client.post(
        "/posts/",
        json={"title": "My First Post", "content": "Hello world!"},
        headers={"Authorization": f"Bearer {author_token}"},
    )
    assert resp.status_code == 201
    data = resp.json()
    assert data["title"] == "My First Post"
    assert data["author"]["username"] == "authoruser"

## ✍️ `test_create_post_as_reader_forbidden`

In [ ]:
def test_create_post_as_reader_forbidden(client, reader_token):
    """❌ 403 · Reader ممنوع من النشر"""
    resp = client.post(
        "/posts/",
        json={"title": "Nope", "content": "Should fail"},
        headers={"Authorization": f"Bearer {reader_token}"},
    )
    assert resp.status_code == 403

## ✍️ `test_create_post_unauthenticated`

In [ ]:
def test_create_post_unauthenticated(client):
    """❌ 401 · بدون توكن"""
    resp = client.post("/posts/", json={"title": "Nope", "content": "Fail"})
    assert resp.status_code == 401

## 🔍 `test_get_single_post`

In [ ]:
def test_get_single_post(client, author_token):
    """✅ 200 · id في الـ response صح"""
    create_resp = client.post(
        "/posts/",
        json={"title": "Single", "content": "Content"},
        headers={"Authorization": f"Bearer {author_token}"},
    )
    post_id = create_resp.json()["id"]
    resp = client.get(f"/posts/{post_id}")
    assert resp.status_code == 200
    assert resp.json()["id"] == post_id

## 🔍 `test_get_nonexistent_post`

In [ ]:
def test_get_nonexistent_post(client):
    """❌ 404 · بوست بـ ID مش موجود"""
    resp = client.get("/posts/9999")
    assert resp.status_code == 404

## ✏️ `test_update_own_post`

In [ ]:
def test_update_own_post(client, author_token):
    """✅ 200 · title اتحدّث"""
    post_id = client.post(
        "/posts/",
        json={"title": "Old Title", "content": "Old content"},
        headers={"Authorization": f"Bearer {author_token}"},
    ).json()["id"]
    resp = client.put(
        f"/posts/{post_id}",
        json={"title": "New Title"},
        headers={"Authorization": f"Bearer {author_token}"},
    )
    assert resp.status_code == 200
    assert resp.json()["title"] == "New Title"

## 🗑️ `test_delete_post_as_admin`

In [ ]:
def test_delete_post_as_admin(client, author_token, admin_token):
    """✅ 204 · Admin يحذف أي بوست"""
    post_id = client.post(
        "/posts/",
        json={"title": "To Delete", "content": "Bye"},
        headers={"Authorization": f"Bearer {author_token}"},
    ).json()["id"]
    resp = client.delete(f"/posts/{post_id}", headers={"Authorization": f"Bearer {admin_token}"})
    assert resp.status_code == 204

## 🗑️ `test_delete_post_as_non_admin_forbidden`

In [ ]:
def test_delete_post_as_non_admin_forbidden(client, author_token):
    """❌ 403 · Author ممنوع من حذف البوستات"""
    post_id = client.post(
        "/posts/",
        json={"title": "No delete", "content": "Stay"},
        headers={"Authorization": f"Bearer {author_token}"},
    ).json()["id"]
    resp = client.delete(f"/posts/{post_id}", headers={"Authorization": f"Bearer {author_token}"})
    assert resp.status_code == 403

## 📄 `test_pagination`

In [ ]:
def test_pagination(client, author_token):
    """✅ total=5 · items=3 · pages=2  (5 بوستات ÷ size=3 = صفحتين)"""
    for i in range(5):
        client.post(
            "/posts/",
            json={"title": f"Post {i}", "content": "Content"},
            headers={"Authorization": f"Bearer {author_token}"},
        )
    resp = client.get("/posts/?page=1&size=3")
    data = resp.json()
    assert data["total"] == 5
    assert len(data["items"]) == 3
    assert data["pages"] == 2

## ▶️ تشغيل الـ Tests

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "test_posts.py", "-v", "--tb=short"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:\n", result.stderr)